In [1]:
path_common_eurobench = '/Users/jorge/Documents/Coding/Projects/nrg-standardization/common_eurobench'

# Characterization

In [2]:
# Import packages from different folders
import sys
sys.path.append(path_common_eurobench)

In [3]:
import pathlib as path
import exploration.exploration_utils as aux
import re

In and out routes

In [4]:
route_in = './data_eurobench'
route_out_dir_csv = path.Path.cwd().joinpath('characterization')
route_out_dir_csv.mkdir(parents=True, exist_ok=True)

Unique columns in the data

In [5]:
pattern_csv = '**/*.csv'
files = path.Path(route_in).glob(pattern_csv)
unique_cols = aux.find_unique_cols_csv(files)
unique_cols

### In this cell a new custom function is created to characterize the data. 
This function will be passed to the function characterize_dir, which will apply it to all the files in the route_in folder. This function will calculate the mean and std of each one of the columns in the dataframe, and will return a dataframe with the mean and std of each one of the columns, and the name of the subject, condition and run.

In [6]:
import characterization.characterization_utils as char
import pandas as pd

# Create empty dataframe
pattern = r'(subject_\d+_cond_\d+_run_\d+)'
pattern_csv = '**/*.csv'
df_transposed = []

# Create custom function to characterize the data
def characterize(df, file, info, events):
    """
    custom function to characterize the data
    :param df: dataframe with the data
    :param file: file to characterize
    :param info: info of the file
    :param events: events of the file
    """
    
    # Find the name of the subject, condition and run
    subject_name = re.match(r'subject_(\d+)', file.name).group(1)
    condition_name = re.match(r'.*cond_(\d+)', file.name).group(1)
    run_name = re.match(r'.*run_(\d+)', file.name).group(1)

    # Create dict with all the info
    dict_curr = {'filename': file.name, 'class': ('PD' if info['affected'] == True else 'HC'),
                     'subject': subject_name, 'condition': condition_name, 'run': run_name}
    
    # Find the mean of each one of the columns in df: this will be the feature to consider
    df_mean = pd.DataFrame()
    for col in df.columns:
        df_mean[col] = [df[col].mean()]
    df_mean['feature'] = 'mean'
    
    # Find the std of each one of the columns in df: this will be the feature to consider
    df_std = pd.DataFrame()
    for col in df.columns:
        df_std[col] = [df[col].std()]
    df_std['feature'] = 'std'
    
    # Concatena df_mean y df_std, now we have two features
    df_feat = pd.concat([df_mean, df_std], ignore_index=True, axis=0)
    
    # Repeat df_dict as many times as rows in df_feat (to be able to concatenate)
    df_dict = pd.DataFrame(dict_curr, index=[0])
    df_dict = pd.concat([df_dict]*df_feat.shape[0], ignore_index=True)
    
    # Concatenate df and df_dict
    df_final = pd.concat([df_feat, df_dict], axis=1)
    
    return df_final
        
# Characterize all files in route_in
char.characterize_dir(route_in, route_out_dir_csv, characterize, pattern_csv, unique_cols, pattern)       

In [7]:
# Concatenate all dataframes into one by reading the contents of the route_out_dir_csv folder
df_all = pd.DataFrame()
for file in path.Path(route_out_dir_csv).glob(f'{characterize.__name__}_*.csv'):
    df = pd.read_csv(file)
    df_all = pd.concat([df_all, df], ignore_index=True, axis=0)
df_all.head()

This is the name of the features in the dataframe

In [8]:
list_all_feats = [col for col in df_all.columns if col not in ['filename', 'class', 'feature', 'subject', 'condition', 'run']]
list_all_feats

### Plotting a qq-plot to check normality of the data

In [9]:
type_plot = 'qq'
aux.plot_features_per_class(df_all, 'class', list_all_feats, grouping_var='feature', save=True, type_plot=type_plot)

### Plotting a boxplot to check distribution of the data

In [10]:
type_plot = 'box'
aux.plot_features_per_class(df_all, 'class', list_all_feats, grouping_var='feature', save=True, type_plot=type_plot)